# 3 — Frozen scGPT embeddings for all cells in the 10 samples

## Question

Can frozen scGPT generate a reusable 512-dimensional embedding for every one
of the same 29,614 cells used by the Scanpy baseline?

Run this notebook on Colab GPU. No cell-type or response label enters scGPT.

## 0. Install one lightweight file-reading dependency

The default Colab kernel needs only `anndata` to verify input/output files.
Scanpy and scGPT are installed later inside the isolated Python 3.11
environment. Let this cell finish; it may take about a minute on a new runtime.

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("anndata") is None:
    print("Installing anndata in the default Colab kernel; please wait...")
    subprocess.run([sys.executable, "-m", "pip", "install", "anndata"], check=True)
print("anndata is available")

In [ ]:
from pathlib import Path
import subprocess, sys
import numpy as np
import anndata as ad
import torch

IN_COLAB = Path("/content").exists()
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/scrna_oncology_fm")
else:
    ROOT = Path.cwd().resolve()
    if ROOT.name == "notebooks": ROOT = ROOT.parent

MODEL_DIR = ROOT / "models/scGPT_human"
OUTPUT = ROOT / "results/embeddings/GSE205335_scgpt_frozen_v1.h5ad"
SCGPT_PYTHON = Path("/content/scgpt-venv/bin/python") if IN_COLAB else Path(sys.executable)
for name in ["best_model.pt", "args.json", "vocab.json"]:
    assert (MODEL_DIR / name).exists(), f"Missing {MODEL_DIR / name}"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if IN_COLAB and DEVICE != "cuda": raise RuntimeError("Select a Colab GPU runtime")
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU")

## 1. Create/reuse the isolated scGPT environment

In [ ]:
needs_install = IN_COLAB and (not SCGPT_PYTHON.exists() or subprocess.run(
    [str(SCGPT_PYTHON), "-c", "import torch, torchtext, scgpt"], capture_output=True).returncode != 0)
if needs_install:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    if not SCGPT_PYTHON.exists():
        subprocess.run(["uv", "python", "install", "3.11"], check=True)
        subprocess.run(["uv", "venv", str(SCGPT_PYTHON.parent.parent), "--python", "3.11"], check=True)
    subprocess.run(["uv", "pip", "install", "--python", str(SCGPT_PYTHON), "torch==2.3.0",
                    "--index-url", "https://download.pytorch.org/whl/cu121"], check=True)
    subprocess.run(["uv", "pip", "install", "--python", str(SCGPT_PYTHON), "numpy<2",
                    "torchtext==0.18.0", "scgpt==0.2.4", "fsspec[http]<=2024.6.1",
                    "scanpy", "anndata", "pandas", "scipy", "ipython"], check=True)

# scGPT imports IPython even in the non-interactive worker. Repair an existing
# environment created by an earlier notebook version without reinstalling it.
if subprocess.run([str(SCGPT_PYTHON), "-c", "import IPython"], capture_output=True).returncode != 0:
    subprocess.run(["uv", "pip", "install", "--python", str(SCGPT_PYTHON), "ipython"], check=True)
probe = subprocess.run([
    str(SCGPT_PYTHON), "-c",
    "import os; os.environ['MPLBACKEND']='Agg'; "
    "import torch; print('torch', torch.__version__); "
    "import torchtext; print('torchtext', torchtext.__version__); "
    "import scgpt; print('scGPT', scgpt.__version__); "
    "print('CUDA', torch.cuda.is_available())",
], capture_output=True, text=True)
print(probe.stdout)
if probe.returncode != 0:
    print("scGPT environment probe failed. Full error:")
    print(probe.stderr)
    raise RuntimeError("scGPT environment is incomplete; use the full error printed above.")
print("scGPT environment is ready")

## 2. Verify cohort and embed all cells

In [ ]:
source = ad.read_h5ad(ROOT / "data/processed/GSE205335_phase1_raw_counts.h5ad", backed="r")
mask = source.obs["core.patient"].astype(str).eq("Core") & source.obs["Tissue origin"].astype(str).eq("Metastatic LN")
assert int(mask.sum()) == 29_614 and source.obs.loc[mask, "Sample"].nunique() == 10
source.file.close()

worker = Path("/content/run_scgpt_embedding.py") if IN_COLAB else ROOT / "scripts/run_scgpt_embedding.py"
worker.write_text('#!/usr/bin/env python3\n"""Generate frozen scGPT embeddings in an isolated Python environment."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport os\nfrom pathlib import Path\n\nos.environ.setdefault("NUMBA_CACHE_DIR", "/tmp/numba_cache")\nos.environ["MPLBACKEND"] = "Agg"\n\nimport numpy as np\nimport scanpy as sc\nimport torch\nimport scgpt as scg\nfrom scgpt.tasks import cell_emb\n\nLABEL_COLUMNS = ["cluster.total", "lineage.total", "lineage.sub", "cluster.sub", "celltype"]\nOBS_TO_SAVE = ["Sample", "Patient", "Platform", "Tissue origin"]\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--root", type=Path, required=True)\n    parser.add_argument("--batch-size", type=int, default=16)\n    parser.add_argument("--random-state", type=int, default=0)\n    parser.add_argument("--device", choices=["cuda", "cpu"], default="cuda")\n    return parser.parse_args()\n\n\ndef force_single_process_dataloader() -> None:\n    """Prevent scGPT 0.2.4 from duplicating its dense matrix across workers."""\n    original = cell_emb.DataLoader\n\n    def safe_dataloader(*args, **kwargs):\n        kwargs["num_workers"] = 0\n        kwargs["pin_memory"] = False\n        return original(*args, **kwargs)\n\n    cell_emb.DataLoader = safe_dataloader\n\n\ndef main() -> None:\n    args = parse_args()\n    root = args.root.resolve()\n    source_path = root / "data/processed/GSE205335_phase1_raw_counts.h5ad"\n    model_dir = root / "models/scGPT_human"\n    output_path = root / "results/embeddings/GSE205335_scgpt_frozen_v1.h5ad"\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n\n    if args.device == "cuda" and not torch.cuda.is_available():\n        raise RuntimeError("CUDA is unavailable in the isolated scGPT process")\n    np.random.seed(args.random_state)\n    torch.manual_seed(args.random_state)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(args.random_state)\n    torch.set_grad_enabled(False)\n\n    adata = sc.read_h5ad(source_path)\n    if adata.shape != (96_505, 33_714):\n        raise ValueError(f"Unexpected source shape: {adata.shape}")\n    cohort_mask = (\n        adata.obs["core.patient"].astype(str).eq("Core")\n        & adata.obs["Tissue origin"].astype(str).eq("Metastatic LN")\n    )\n    work = adata[cohort_mask].copy()\n    if work.n_obs != 29_614 or work.obs["Sample"].nunique() != 10:\n        raise ValueError(\n            f"Expected 29,614 cells from 10 site-matched samples; got "\n            f"{work.n_obs:,} cells from {work.obs[\'Sample\'].nunique()} samples"\n        )\n    work.obs.drop(columns=LABEL_COLUMNS, inplace=True)\n    work.var["gene_name"] = work.var_names.astype(str)\n    del adata\n\n    device_name = torch.cuda.get_device_name(0) if args.device == "cuda" else "CPU"\n    print(f"Embedding {work.n_obs:,} cells on {device_name}")\n    force_single_process_dataloader()\n    embedding = scg.tasks.embed_data(\n        work,\n        model_dir,\n        gene_col="gene_name",\n        obs_to_save=OBS_TO_SAVE,\n        batch_size=args.batch_size,\n        device=args.device,\n        use_fast_transformer=False,\n        return_new_adata=True,\n    )\n    if embedding.n_obs != work.n_obs or not embedding.obs_names.equals(work.obs_names):\n        raise ValueError("Embedding cells do not match the selected input")\n    if not np.isfinite(embedding.X).all():\n        raise ValueError("Embedding contains non-finite values")\n    embedding.write_h5ad(output_path, compression="gzip")\n    print(f"Wrote {output_path} with shape {embedding.shape}")\n\n\nif __name__ == "__main__":\n    main()\n')
RUN_EMBEDDING = True
if RUN_EMBEDDING:
    subprocess.run([str(SCGPT_PYTHON), str(worker), "--root", str(ROOT), "--batch-size", "16",
                    "--random-state", "0", "--device", DEVICE], check=True)

embedding = ad.read_h5ad(OUTPUT)
assert embedding.shape == (29_614, 512)
assert embedding.obs["Sample"].nunique() == 10
assert np.isfinite(embedding.X).all()
print("Complete scGPT embedding:", embedding.shape)
print("Saved:", OUTPUT)